# Multi-Horizon Benchmark Analysis and Visualisation

**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

This notebook aggregates results from **all trained models** across **all horizons**  
and generates the publication-quality comparison tables and figures required  
for the research paper.

Run this **after** completing both:
- `01_NiOA_DRNN_Training.ipynb` for every horizon
- `02_Benchmark_Classical_ML.ipynb` and `03_Benchmark_Deep_Learning.ipynb`


In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for _p in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'core')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from core.config import FORECAST_HORIZONS, BENCHMARK_DIR, NIOA_RESULTS_DIR, RESULTS_DIR
from core.config import *
from benchmarking.utils.metrics import build_comparison_table

ANALYSIS_DIR = os.path.join(RESULTS_DIR, 'analysis')
os.makedirs(ANALYSIS_DIR, exist_ok=True)

sns.set(style='whitegrid', palette='tab10')
print(f'Project root : {PROJECT_ROOT}')
print('Analysis notebook ready.')

## 1 · Build Full Results Table

In [ ]:
all_rows = []

for h in FORECAST_HORIZONS:
    try:
        tbl = build_comparison_table(BENCHMARK_DIR, h)
        tbl['Horizon_s'] = h
        all_rows.append(tbl)
    except FileNotFoundError:
        print(f'  No benchmark results for k={h} — skipping.')

if not all_rows:
    print('No results found. Run notebooks 01-03 first.')
else:
    results = pd.concat(all_rows, ignore_index=True)

    pivot_mae = results.pivot(index='Model', columns='Horizon_s', values='MAE')
    print('\nMAE Comparison Table (benchmark models)')
    print(pivot_mae.to_string(float_format='{:.6f}'.format))

    results.to_csv(os.path.join(ANALYSIS_DIR, 'full_results.csv'), index=False)
    pivot_mae.to_csv(os.path.join(ANALYSIS_DIR, 'mae_pivot.csv'))
    print(f'\nSaved to : {ANALYSIS_DIR}')

## 2 · MAE vs Horizon Plot

In [ ]:
if 'results' in dir() and not results.empty:
    available_h = sorted(results['Horizon_s'].unique())

    fig, ax = plt.subplots(figsize=(10, 5))
    for model_name, grp in results.groupby('Model'):
        ax.plot(grp['Horizon_s'], grp['MAE'],
                marker='o', linewidth=2, markersize=7, label=model_name)
    ax.set_xscale('log')
    ax.set_xticks(available_h)
    ax.set_xticklabels([str(h) for h in available_h])
    ax.set_xlabel('Forecast Horizon k (seconds)')
    ax.set_ylabel('MAE (kWh)')
    ax.set_title('MAE vs Forecast Horizon — All Benchmark Models')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, 'mae_vs_horizon.png'), dpi=200, bbox_inches='tight')
    plt.close('all')
    print(f'Saved → {os.path.join(ANALYSIS_DIR, "mae_vs_horizon.png")}')

## 3 · R² vs Horizon Plot

In [ ]:
if 'results' in dir() and not results.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    for model_name, grp in results.groupby('Model'):
        ax.plot(grp['Horizon_s'], grp['R2'],
                marker='s', linewidth=2, markersize=7, label=model_name)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax.set_xscale('log')
    ax.set_xticks(available_h)
    ax.set_xticklabels([str(h) for h in available_h])
    ax.set_xlabel('Forecast Horizon k (seconds)')
    ax.set_ylabel('R²')
    ax.set_title('R² vs Forecast Horizon — All Benchmark Models')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, 'r2_vs_horizon.png'), dpi=200, bbox_inches='tight')
    plt.close('all')
    print(f'Saved → {os.path.join(ANALYSIS_DIR, "r2_vs_horizon.png")}')

## 4 · Per-Horizon Grouped Bar Chart (MAE)

In [ ]:
if 'results' in dir() and not results.empty:
    _horizons = sorted(results['Horizon_s'].unique())
    _n = len(_horizons)
    fig, axes = plt.subplots(1, _n, figsize=(5 * _n, 5), sharey=False)
    if _n == 1:
        axes = [axes]

    for ax, h in zip(axes, _horizons):
        sub = results[results['Horizon_s'] == h].sort_values('MAE')
        ax.barh(sub['Model'], sub['MAE'],
                color=sns.color_palette('tab10', len(sub)))
        ax.set_title(f'k = {h} s')
        ax.set_xlabel('MAE (kWh)')
        ax.invert_yaxis()

    plt.suptitle('MAE Comparison — All Horizons', fontsize=14, y=1.02)
    plt.tight_layout()
    _path = os.path.join(ANALYSIS_DIR, 'mae_bar_all_horizons.png')
    plt.savefig(_path, dpi=200, bbox_inches='tight')
    plt.close('all')
    print(f'Saved → {_path}')
    print(f'\nAll analysis figures in : {ANALYSIS_DIR}')